# Lecture 10C.2 — openSMILE eGeMAPS: Standardized Acoustic Features


## Goal

Use openSMILE (via the Python `opensmile` package) to extract:

- eGeMAPS (a compact, widely used feature set)
- low-level descriptors + functionals

This gives you a **high-coverage baseline** for emotion, health, and paralinguistics tasks.


## 1) Setup


In [7]:
import os, json, re, math
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Optional audio playback
try:
    from IPython.display import Audio, display
    HAS_IPY_AUDIO = True
except Exception:
    HAS_IPY_AUDIO = False

# ---------------- Project paths (shared manifest workflow) ----------------
PROJECT_ROOT = Path.cwd() / "EE519_L10C_Project"
REC_DIR = PROJECT_ROOT / "recordings"
FIG_DIR = PROJECT_ROOT / "figures"
RES_DIR = PROJECT_ROOT / "results"
FEAT_DIR = PROJECT_ROOT / "features"
MANIFEST_PATH = PROJECT_ROOT / "manifest.json"

for d in [REC_DIR, FIG_DIR, RES_DIR, FEAT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

def load_manifest(path=MANIFEST_PATH):
    if path.exists():
        return json.loads(path.read_text())
    return {"course":"EE519","module":"Lecture10C","created_utc":None,"clips":[],"splits":{}}

def save_manifest(manifest, path=MANIFEST_PATH):
    if manifest.get("created_utc") is None:
        manifest["created_utc"] = str(np.datetime64("now"))
    path.write_text(json.dumps(manifest, indent=2))
    print("Saved manifest:", path)

def save_fig(fig, name, dpi=150):
    out = FIG_DIR / name
    fig.savefig(out, dpi=dpi, bbox_inches="tight")
    print("Saved:", out)
    return out

import wave
def read_wav(path: Path):
    with wave.open(str(path), "rb") as wf:
        fs = wf.getframerate()
        n = wf.getnframes()
        x = np.frombuffer(wf.readframes(n), dtype=np.int16).astype(np.float32) / 32768.0
    return fs, x

def peak_normalize(x, target=0.98):
    m = np.max(np.abs(x)) + 1e-12
    return (x / m) * target

def play_audio(x, fs, label="audio"):
    if not HAS_IPY_AUDIO:
        print("(Audio playback not available)", label)
        return
    display(Audio(x, rate=fs))

def list_clips(manifest):
    for i,c in enumerate(manifest.get("clips", [])):
        print(f"[{i}] {c.get('label',''):14s}  {c.get('filename','')}  fs={c.get('fs','?')}  notes={c.get('notes','')}")

def get_segments(clip):
    # We reuse analysis_segments from earlier threads if present
    return clip.get("selections", {}).get("analysis_segments", {})

manifest = load_manifest()
print("Project root:", PROJECT_ROOT)
print("Clips:", len(manifest.get('clips', [])))
list_clips(manifest)


Project root: c:\Users\K\Documents\usc\ee519\ee519-lecture\lecture10c\EE519_L10C_Project
Clips: 12
[0] vowel_a         student10B_vowel_a.wav  fs=16000  notes=steady /a/
[1] vowel_i         student10B_vowel_i.wav  fs=16000  notes=steady /i/
[2] fricative_s     student10B_fricative_s.wav  fs=16000  notes=steady /s/
[3] sentence        student10B_sentence.wav  fs=16000  notes=short sentence
[4] vowel_a         student10B_vowel_a.wav  fs=16000  notes=steady /a/
[5] vowel_i         student10B_vowel_i.wav  fs=16000  notes=steady /i/
[6] fricative_s     student10B_fricative_s.wav  fs=16000  notes=steady /s/
[7] sentence        student10B_sentence.wav  fs=16000  notes=short sentence
[8] vowel_a         student10B_vowel_a.wav  fs=16000  notes=steady /a/
[9] vowel_i         student10B_vowel_i.wav  fs=16000  notes=steady /i/
[10] fricative_s     student10B_fricative_s.wav  fs=16000  notes=steady /s/
[11] sentence        student10B_sentence.wav  fs=16000  notes=short sentence


## 2) Install / import opensmile (with fallback)

If `opensmile` isn’t installed, you can still read this notebook and run later.


In [2]:
#pip install --upgrade pip setuptools wheel

In [12]:
#pip install opensmile

In [9]:
#brew install opensmile

In [10]:
#pip install --upgrade pip setuptools wheel

In [14]:
#python3 -m pip install --upgrade pip setuptools wheel

In [8]:
HAS_SMILE = True
try:
    import opensmile
except Exception as e:
    HAS_SMILE = False
    print("opensmile not available:", repr(e))
    print("Install: pip install opensmile")


## 3) Extract eGeMAPS per segment

We will:
- read each segment from manifest
- run openSMILE
- save one row per segment


In [9]:
manifest = load_manifest()

# Build segment list
items = []
for ci, clip in enumerate(manifest.get("clips", [])):
    for seg_name, sel in get_segments(clip).items():
        items.append((ci, seg_name, sel))

print("Segments:", len(items))

if not HAS_SMILE:
    print("Skipping extraction because opensmile is missing.")
else:
    smile = opensmile.Smile(
        feature_set=opensmile.FeatureSet.eGeMAPSv02,
        feature_level=opensmile.FeatureLevel.Functionals,
    )

    feat_rows = []
    for ci, seg_name, sel in items:
        clip = manifest["clips"][ci]
        wav_path = REC_DIR / clip["filename"]
        fs, x = read_wav(wav_path); x = peak_normalize(x)
        xseg = x[int(sel["s0"]):int(sel["s1"])]
        if len(xseg) < int(0.1*fs):
            # too short for stable statistics
            continue
        # opensmile can process arrays directly
        df = smile.process_signal(xseg, fs)
        # flatten to dict
        r = df.iloc[0].to_dict()
        r.update({"clip_idx":ci, "segment":seg_name, "label":clip.get("label",""), "filename":clip.get("filename","")})
        feat_rows.append(r)

    smile_df = pd.DataFrame(feat_rows)
    display(smile_df.head())

    out_csv = FEAT_DIR / "opensmile_eGeMAPS_segments.csv"
    smile_df.to_csv(out_csv, index=False)
    print("Saved:", out_csv)


Segments: 6


,F0semitoneFrom27.5Hz_sma3nz_amean,F0semitoneFrom27.5Hz_sma3nz_stddevNorm,F0semitoneFrom27.5Hz_sma3nz_percentile20.0,F0semitoneFrom27.5Hz_sma3nz_percentile50.0,F0semitoneFrom27.5Hz_sma3nz_percentile80.0,F0semitoneFrom27.5Hz_sma3nz_pctlrange0-2,F0semitoneFrom27.5Hz_sma3nz_meanRisingSlope,F0semitoneFrom27.5Hz_sma3nz_stddevRisingSlope,F0semitoneFrom27.5Hz_sma3nz_meanFallingSlope,F0semitoneFrom27.5Hz_sma3nz_stddevFallingSlope,...,VoicedSegmentsPerSec,MeanVoicedSegmentLengthSec,StddevVoicedSegmentLengthSec,MeanUnvoicedSegmentLength,StddevUnvoicedSegmentLength,equivalentSoundLevel_dBp,clip_idx,segment,label,filename
0,19.326334,0.006981,19.224199,19.290649,19.368874,0.144674,2.461863,0.608397,2.629618,1.258209,...,2.325582,0.42,0.000000,0.000,0.00000,-13.976613,0,vowel_mid,vowel_a,student10B_vowel_a.wav
1,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.00,0.000000,0.430,0.00000,-15.456470,2,fricative_mid,fricative_s,student10B_fricative_s.wav
2,21.039663,0.031176,20.686052,21.083149,21.504265,0.818213,100.973511,38.600082,16.213900,4.799537,...,4.761905,0.08,0.043205,0.110,0.04899,-16.453550,3,seg1,sentence,student10B_sentence.wav
3,21.039663,0.031176,20.686052,21.083149,21.504265,0.818213,100.973511,38.600082,16.213900,4.799537,...,4.761905,0.08,0.043205,0.110,0.04899,-16.453550,3,vowel_mid,sentence,student10B_sentence.wav
4,19.309488,0.018935,19.077705,19.253090,19.684847,0.607141,1.572126,3.997797,5.733402,0.000000,...,2.272727,0.26,0.000000,0.075,0.02500,-15.185431,3,fricative_mid,sentence,student10B_sentence.wav


Saved: c:\Users\K\Documents\usc\ee519\ee519-lecture\lecture10c\EE519_L10C_Project\features\opensmile_eGeMAPS_segments.csv


## 4) Quick exploration: which features separate vowel vs fricative?

We’ll:
- join with the segment split table (if created in 10C.0)
- compute simple correlations / separations


In [10]:
split_csv = RES_DIR / "L10C_segment_split.csv"
if not split_csv.exists():
    print("No split table found. Run 10C.0 first.")
elif not ('smile_df' in globals()):
    print("No openSMILE table in memory. Run extraction above (requires opensmile).")
else:
    split_df = pd.read_csv(split_csv)
    # merge on clip_idx+segment
    merged = split_df.merge(smile_df, on=["clip_idx","segment","label","filename"], how="inner")
    print("Merged rows:", len(merged))

    # pick a few interpretable eGeMAPS examples (names may vary slightly)
    cols = [c for c in merged.columns if any(k in c.lower() for k in ["f0", "jitter", "shimmer", "h1", "alpha", "slope", "loudness", "mfcc"])]
    cols = cols[:12]
    display(merged[["group","split"] + cols].head())

    # simple group means
    means = merged.groupby("group")[cols].mean()
    display(means)


Merged rows: 6


,group,split,F0semitoneFrom27.5Hz_sma3nz_amean,F0semitoneFrom27.5Hz_sma3nz_stddevNorm,F0semitoneFrom27.5Hz_sma3nz_percentile20.0,F0semitoneFrom27.5Hz_sma3nz_percentile50.0,F0semitoneFrom27.5Hz_sma3nz_percentile80.0,F0semitoneFrom27.5Hz_sma3nz_pctlrange0-2,F0semitoneFrom27.5Hz_sma3nz_meanRisingSlope,F0semitoneFrom27.5Hz_sma3nz_stddevRisingSlope,F0semitoneFrom27.5Hz_sma3nz_meanFallingSlope,F0semitoneFrom27.5Hz_sma3nz_stddevFallingSlope,loudness_sma3_amean,loudness_sma3_stddevNorm
0,vowel_voiced,train,19.326334,0.006981,19.224199,19.290649,19.368874,0.144674,2.461863,0.608397,2.629618,1.258209,1.837309,0.104722
1,fricative_unvoiced,test,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,2.222774,0.276763
2,fricative_unvoiced,train,21.039663,0.031176,20.686052,21.083149,21.504265,0.818213,100.973511,38.600082,16.213900,4.799537,1.103576,0.374791
3,vowel_voiced,test,21.039663,0.031176,20.686052,21.083149,21.504265,0.818213,100.973511,38.600082,16.213900,4.799537,1.103576,0.374791
4,fricative_unvoiced,train,19.309488,0.018935,19.077705,19.253090,19.684847,0.607141,1.572126,3.997797,5.733402,0.000000,1.554594,0.454109


,F0semitoneFrom27.5Hz_sma3nz_amean,F0semitoneFrom27.5Hz_sma3nz_stddevNorm,F0semitoneFrom27.5Hz_sma3nz_percentile20.0,F0semitoneFrom27.5Hz_sma3nz_percentile50.0,F0semitoneFrom27.5Hz_sma3nz_percentile80.0,F0semitoneFrom27.5Hz_sma3nz_pctlrange0-2,F0semitoneFrom27.5Hz_sma3nz_meanRisingSlope,F0semitoneFrom27.5Hz_sma3nz_stddevRisingSlope,F0semitoneFrom27.5Hz_sma3nz_meanFallingSlope,F0semitoneFrom27.5Hz_sma3nz_stddevFallingSlope,loudness_sma3_amean,loudness_sma3_stddevNorm
group,,,,,,,,,,,,
fricative_unvoiced,14.914660,0.017261,14.710366,14.897332,15.218490,0.508124,26.029441,11.648919,6.920176,1.199884,1.608885,0.389943
vowel_voiced,20.182999,0.019078,19.955126,20.186899,20.436569,0.481443,51.717687,19.604239,9.421759,3.028873,1.470443,0.239756


## Reflection questions

1) Why are standardized sets like eGeMAPS useful for benchmarking?  
2) Which openSMILE features would you expect to correlate with “loud vs soft”?  
3) Why do “functionals” (mean/std/percentiles) matter for variable-length segments?


### Answers

1. They can ensure consistency across multiple experiments.
2. Perceptual loudness and energy likely correlate with loud vs soft.
3. Functionals help us understand the distribution of the data which is useful for classification.

## What’s next
- **10C.3** Self-supervised embeddings (wav2vec2/HuBERT): feature learning instead of hand-crafting.
